Load schema definitions and config

In [0]:
%run ../config/config

In [0]:
dbutils.widgets.text("batch_id","")
batch_id=dbutils.widgets.get("batch_id")

In [0]:
bronze_table = f"{catalog}.{bronze_schema}.stations"
silver_table = f"{catalog}.{silver_schema}.stations"

Read bronze Delta table into a DataFrame

In [0]:
bronze_stations_df=(
    spark.read
    .format("delta")
    .table(bronze_table)
)

Drop null records

In [0]:
bronze_stations_df=bronze_stations_df.dropna()

Drop duplicated records

In [0]:
bronze_stations_df=bronze_stations_df.dropDuplicates()

Separate display_airport_city_name_full column into separate columns

In [0]:
from pyspark.sql import functions as F

silver_stations_df=(
    bronze_stations_df
    .withColumn(
        "display_airport_city_name_full", F.split(F.col("display_airport_city_name_full"),",")[0]
    )
)

Select and rename columns for clarity and unification

In [0]:
silver_stations_df=(
    silver_stations_df
    .select(
        "airport_id",
        "airport",
        "display_airport_name",
        F.col("display_airport_city_name_full").alias("airport_city"),
        F.col("airport_state_name").alias("airport_state"),
        "airport_state_code",
        "latitude",
        "longitude",
        "elevation",
        "icao",
        "iata",
        "faa",
        "mesonet_station",
        "batch_id"
    )
)

Add created and updated timestamp columns

In [0]:
silver_stations_df=(
    silver_stations_df
    .withColumns({
        "created_timestamp":F.current_timestamp(),
        "updated_timestamp":F.current_timestamp()
    })
)

Write DataFrame to silver Delta table 

Subsequent runs merge new batch into existing table, only updating records from a newer or equal batch to avoid reprocessing

In [0]:
from pyspark.sql import Window

window = Window.partitionBy("airport_id").orderBy(
    F.col("batch_id").desc()
)

silver_stations_df = (
    silver_stations_df
    .withColumn("_rn", F.row_number().over(window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

if not spark.catalog.tableExists(silver_table):
    silver_stations_df_write=(
        silver_stations_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
    )

else:
    from delta.tables import DeltaTable

    delta_table=DeltaTable.forName(spark, silver_table)
    (
        delta_table.alias("t")
        .merge(
            silver_stations_df.alias("s"),
            "t.airport_id = s.airport_id"

        )
        .whenMatchedUpdate(
            condition="s.batch_id >= t.batch_id",
            set={
                "airport" :"s.airport",
                "display_airport_name" : "s.display_airport_name",
                "airport_city": "s.airport_city",
                "airport_state" : "s.airport_state",
                "airport_state_code" : "s.airport_state_code",
                "latitude": "s.latitude",
                "longitude": "s.longitude",
                "elevation" : "s.elevation",
                "icao": "s.icao",
                "iata": "s.iata",
                "faa": "s.faa",
                "mesonet_station": "s.mesonet_station",
                "batch_id": "s.batch_id",
                "updated_timestamp": "s.updated_timestamp"
            }

        )
        .whenNotMatchedInsertAll()
        .execute()
    )